In [1]:
import pandas as pd  
import numpy as np  
import os  
import re  
import glob  
import io
from datetime import datetime  
import msoffcrypto
import warnings
from dateutil.relativedelta import relativedelta

warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')

In [ ]:
pc_folder_path = 'L:/2026 Pilar Plant Files/Position Control'

# --- adjust date ---  
file_date = '2026-08-29'
# -------------------

# get all matching files  
pc_files = glob.glob(os.path.join(pc_folder_path, f'*{file_date}*.xlsx'))

print(f"Found {len(pc_files)} files matching date {file_date}")

site_pc_list = []

for file in pc_files:  
    filename = os.path.basename(file)  
      
    # extract site (first 5 characters of filename)  
    site = filename[:5]  
      
    # read second tab, skip first 12 rows  
    file_data = pd.read_excel(file, sheet_name='PositionDetail', skiprows=12)  
      
    # add site and date columns  
    file_data['BU'] = site  
    file_data['Date'] = file_date  

    f_file_data = file_data[['Rpt Dept', 'Rpt Dept Desc', 'Job Code', 'Jobcode Title', 'Job Function / Family', 'Posn Number', 
                            'Posn Status', 'Filled/ Open', 'Incumbent Name', 'Emplid', 'Incumbent Status',
                            'Reg/ Temp', 'Posn Type', 'Filled Hrs', 'Filled FTE', 'Open Hrs', 'Open FTE', 'BU', 'Date']]

    fc_file_data = f_file_data[f_file_data['Rpt Dept'].notna()]
    
    site_pc_list.append(fc_file_data)
    print(f"Processed: {filename}")

# combine all into one dataframe  
if site_pc_list:  
    position_control = pd.concat(site_pc_list, ignore_index=True)  
    print(f"\nTotal rows: {len(position_control)}")  
else:  
    print(f"No files found for date: {file_date}")

Found 3 files matching date 2026-08-29
Processed: GVCCC POSNEXEC GVCCC_EXEC 2026-08-29.xlsx
Processed: LENOX POSNEXEC LENOX_EXEC 2026-08-29.xlsx
Processed: MEETH POSNEXEC MEETH_EXEC 2026-08-29.xlsx

Total rows: 6272


In [ ]:
position_control['filled_active'] = np.where(position_control['Incumbent Status'] == 'A', position_control['Filled FTE'], 0) 
position_control['loa'] = np.where(position_control['Incumbent Status'].isin(['L', 'P']), position_control['Filled FTE'], 0)
position_control['open'] = np.where(position_control['Filled/ Open'] == 'Open', position_control['Open FTE'], 0)
position_control['total_ftes'] = position_control['filled_active'] + position_control['loa'] + position_control['open']

pc_values = position_control[position_control['total_ftes'] != 0].reset_index(drop=True)
pc_full = pc_values[['BU', 'Date', 'Rpt Dept', 'Rpt Dept Desc', 'Job Code', 'Jobcode Title',
                         'filled_active', 'loa', 'open', 'total_ftes']]

display(pc_full)

,BU,Date,Rpt Dept,Rpt Dept Desc,Job Code,Jobcode Title,filled_active,loa,open,total_ftes
0,GVCCC,2026-08-29,74002500.0,Administration,111637.0,"Site Leader, Greenwich Village ASC",0.1,0.0,0.0,0.1
1,GVCCC,2026-08-29,74002500.0,Administration,115816.0,"VP, CMO, NHGV",0.8,0.0,0.0,0.8
2,GVCCC,2026-08-29,74002500.0,Administration,200696.0,"Director, Clinical Care",1.0,0.0,0.0,1.0
3,GVCCC,2026-08-29,74002500.0,Administration,200697.0,"Director, Patient Care",1.0,0.0,0.0,1.0
4,GVCCC,2026-08-29,74002500.0,Administration,201068.0,"Manager, Operations",1.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...
5019,MEETH,2026-08-29,29691010.0,Sup Svc-Dietary,108135.0,Dietary Worker,1.0,0.0,0.0,1.0
5020,MEETH,2026-08-29,29691010.0,Sup Svc-Dietary,202519.0,"Manager, Food & Dining Services",1.0,0.0,0.0,1.0
5021,MEETH,2026-08-29,29692010.0,Sup Svc-Security,201608.0,"Senior Manager, Security",1.0,0.0,0.0,1.0
5022,MEETH,2026-08-29,29692010.0,Sup Svc-Security,904094.0,"Senior Director, Safety",0.1,0.0,0.0,0.1


In [ ]:
# pull xwalk reference file
xwalk = pd.read_excel("C:/Users/kbixby/OneDrive - Northwell Health/Scripts/fte/dept_jc_lookup_table.xlsx")

# check for duplicates in crosswalk  
dupes = xwalk[xwalk.duplicated(subset=['dept', 'jc'], keep=False)]
# print dupes if found
if not dupes.empty:  
    print(f"WARNING: {len(dupes)} duplicate dept/jc rows found in crosswalk:")  
    display(dupes.sort_values(['dept', 'jc']))

pc_depts = pd.merge(xwalk[['vp', 'director', 'dept', 'dept_desc']], pc_values, how='right', left_on='dept', right_on='Rpt Dept')

# remove corporate retained and employee health services
pc_depts_f = pc_depts[~pc_depts['Rpt Dept Desc'].str.contains('Corporate Retained|Corp Retained|Employee Health Svcs', case=False, na=False)]  


In [52]:
req_path_name = 'L:/2026 Pilar Plant Files/Requisition Reports/'
req_file_name = 'NYC Req Reports 9.3.2026.xlsx'
pending_req_tab = 'Pending'
approved_req_tab = 'Accepted'

file = req_path_name + req_file_name

# read two tabs into separate dataframes  
pending_reqs = pd.read_excel(file, sheet_name=pending_req_tab) 
approved_reqs = pd.read_excel(file, sheet_name=approved_req_tab)   


approved_reqs_site_spec = approved_reqs[approved_reqs['BUSINESS UNIT'].isin(['LENOX', 'MEETH', 'GVCCC'])]
approved_reqs_full = approved_reqs_site_spec[['BUSINESS UNIT', 'DEPARTMENT NUMBER', 'DEPARTMENT NAME', 'JOB CODE', 'JOB TITLE', 
                                           'REQ. IDENTIFIER', 'FTE']].drop_duplicates()


pending_reqs_site_spec = pending_reqs[pending_reqs['BUSINESS UNIT'].isin(['LENOX', 'MEETH', 'GVCCC'])]
pending_reqs_full = pending_reqs_site_spec[['BUSINESS UNIT', 'DEPARTMENT NUMBER', 'DEPARTMENT NAME', 'JOB CODE', 'JOB TITLE', 
                                         'REQ. IDENTIFIER', 'FTE', 'NEW REPLACE']].drop_duplicates()

In [54]:
display(pc_full.head())
display(approved_reqs_full.head())
display(pending_reqs_full.head())

,BU,Date,Rpt Dept,Rpt Dept Desc,Job Code,Jobcode Title,filled_active,loa,open,total_ftes
0,GVCCC,2026-08-29,74002500.0,Administration,111637.0,"Site Leader, Greenwich Village ASC",0.1,0.0,0.0,0.1
1,GVCCC,2026-08-29,74002500.0,Administration,115816.0,"VP, CMO, NHGV",0.8,0.0,0.0,0.8
2,GVCCC,2026-08-29,74002500.0,Administration,200696.0,"Director, Clinical Care",1.0,0.0,0.0,1.0
3,GVCCC,2026-08-29,74002500.0,Administration,200697.0,"Director, Patient Care",1.0,0.0,0.0,1.0
4,GVCCC,2026-08-29,74002500.0,Administration,201068.0,"Manager, Operations",1.0,0.0,0.0,1.0


,BUSINESS UNIT,DEPARTMENT NUMBER,DEPARTMENT NAME,JOB CODE,JOB TITLE,REQ. IDENTIFIER,FTE
521,GVCCC,74002502,Access Services,110936,ED Associate (GV),155108,0.4
522,GVCCC,74002502,Access Services,110936,ED Associate (GV),177012,0.4
523,GVCCC,74002520,OP Svc - Emergency Services,200095,"Assistant Manager, Patient Care",165419,1.0
524,GVCCC,74002520,OP Svc - Emergency Services,108102,RN,169320,1.0
525,GVCCC,74002520,OP Svc - Emergency Services,108102,RN,169604,1.0


,BUSINESS UNIT,DEPARTMENT NUMBER,DEPARTMENT NAME,JOB CODE,JOB TITLE,REQ. IDENTIFIER,FTE,NEW REPLACE
921,LENOX,15600055,Consulting Services,202242,"Director, Operations",190161,1.000000,REPLACE
922,LENOX,15600126,Executive Health,201273,Optometrist,197240,0.026667,NEW
923,LENOX,15600140,Patient Care Management,115682,MSW-Social Worker,197460,1.000000,REPLACE
924,LENOX,15601045,Patient Experience,201018,"Manager, Clinical Program",195604,1.000000,REPLACE
925,LENOX,15610030,NRSG - Float,108109,Registered Nurse (RN)- Float Pool Med/Surg,196995,1.000000,REPLACE


In [58]:
pc_app_reqs = pd.merge(pc_full, approved_reqs_full, how='outer', left_on=['Rpt Dept', 'Job Code'], right_on=['DEPARTMENT NUMBER', 'JOB CODE'])
pc_app_reqs['dept_id'] = np.where(pc_app_reqs['Rpt Dept'].notna(), pc_app_reqs['Rpt Dept'], pc_app_reqs['DEPARTMENT NUMBER'])
pc_app_reqs['dept_desc'] = np.where(pc_app_reqs['Rpt Dept Desc'].notna(), pc_app_reqs['Rpt Dept Desc'], pc_app_reqs['DEPARTMENT NAME'])

pc_app_reqs['bu'] = np.where(pc_app_reqs['BU'].notna(), pc_app_reqs['BU'], pc_app_reqs['BUSINESS UNIT'])  

pc_app_reqs['job_code'] = np.where(pc_app_reqs['Job Code'].notna(), pc_app_reqs['Job Code'], pc_app_reqs['JOB CODE'])  
pc_app_reqs['job_desc'] = np.where(pc_app_reqs['Jobcode Title'].notna(), pc_app_reqs['Jobcode Title'], pc_app_reqs['JOB TITLE'])  

pc_app_reqs_merged = pc_app_reqs[['bu', 'dept_id', 'dept_desc', 'job_code', 'job_desc', 'filled_active', 'loa', 'open', 'total_ftes', 'REQ. IDENTIFIER', 'FTE']]

display(pc_app_reqs_merged.head())

,bu,dept_id,dept_desc,job_code,job_desc,filled_active,loa,open,total_ftes,REQ. IDENTIFIER,FTE
0,LENOX,15600000.0,Hospital Administration,103442.0,"VP, Marketing",0.5,0.0,0.0,0.5,NaN,NaN
1,LENOX,15600000.0,Hospital Administration,107015.0,Site CMIO,1.0,0.0,0.0,1.0,NaN,NaN
2,LENOX,15600000.0,Hospital Administration,200118.0,Associate Biostatistician,0.5,0.0,0.0,0.5,NaN,NaN
3,LENOX,15600000.0,Hospital Administration,200222.0,"AVP, IT&S Strategic Planning",0.9,0.0,0.0,0.9,NaN,NaN
4,LENOX,15600000.0,Hospital Administration,201068.0,"Manager, Operations",0.0,0.0,1.0,1.0,184445.0,1.0
